# Access metadata

In [17]:
import pandas as pd
import urllib.parse
import re
import requests
import json
import hashlib

!pip install rapidfuzz
from rapidfuzz import fuzz, process

# Spreadsheet IDs
spreadsheet_id = '1CC4sbzh8EtqYs16JSfTR9kUoNU9XfeJQpu3qC7ZWht0'
encoded_sheet_name_cat = urllib.parse.quote('Zeri CATALOGHI')
encoded_sheet_name_asta = urllib.parse.quote('Zeri EVENTO ASTA')
encoded_sheet_name_manifest = urllib.parse.quote('HISTORICA_MANIFEST')
encoded_sheet_name_publishers = urllib.parse.quote('PUBLISHERS')

GID_cat = "1940703370"
# CSV
url_cat = f'https://docs.google.com/spreadsheets/d/{spreadsheet_id}/export?format=csv&gid={GID_cat}'
#url_cat = f'https://docs.google.com/spreadsheets/d/{spreadsheet_id}/gviz/tq?tqx=out:csv&sheet={encoded_sheet_name_cat}'
url_asta = f'https://docs.google.com/spreadsheets/d/{spreadsheet_id}/gviz/tq?tqx=out:csv&sheet={encoded_sheet_name_asta}'
url_manifest = f'https://docs.google.com/spreadsheets/d/{spreadsheet_id}/gviz/tq?tqx=out:csv&sheet={encoded_sheet_name_manifest}'
url_publishers = f'https://docs.google.com/spreadsheets/d/{spreadsheet_id}/gviz/tq?tqx=out:csv&sheet={encoded_sheet_name_publishers}'

# DF
df = pd.read_csv(url_cat, dtype={'INVENTARIO': str}) # , header=[0, 1]
df_asta = pd.read_csv(url_asta, header=[0, 1])
df_manifest = pd.read_csv(url_manifest)
df_publishers = pd.read_csv(url_publishers)

In [18]:
import requests
r = requests.get(url_cat)
print(r.text[:2000])

INVENTARIO,COLLOCAZIONE,TITOLO,ALTRITITOLI,AUTORE,BANDITORE,AUTORE SECONDARIO,ENTE AUTORE,ENTE AUTORE SECONDARIO,DATA,LINGUA,SECOLO,PERIODO STORICO,DIMENSIONI,LUOGO + EDITORE + DATA,FA PARTE DI,DESCRIZIONE2,LUOGO,Catalogo Fondazione Zeri,Altri Collegamenti,Tipo di supporto,Tipo di oggetto,Materiale e tecnica,Vendita all'asta,Fondo,Progetto,Livello bibliografico,Tipologia del materiale,Tipologia COAR,Provider,Data provider,Licenza,Detentore dei diritti
83625,CA 17 1879 0428,"Catalogue de tableaux de premier ordre formant la collection de M. Fréd. Reiset ... Vente Hotel Drouot, le lundi 28 avril 1879 / Hotel Drouot ; Commissaire-Priseur Ch. Pillet",,,"Pillet, Charles",,Hôtel Drouot,,1879,French,19th Century A.D.,,"39 p., [19] carte di tav. : ill. ; 31 cm","[S.l. : s.n., 1879]",,Fusione di + campi: TIPI_OGGETTI_VENDUTI + COLLEZIONISTI_CONCATENATI + DATA ASTA + LUOGODIVENDITA,,,,paper (fiber product),books,printing techniques,"Asta Hôtel Drouot, 28-04-1879",Francia,,,,,,,,
83626; B 15101,C

In [19]:
print(list(df_asta.columns))

[('ID_CATALOGO Gestionale Zeri', 'Unnamed: 0_level_1'), ('INVENTARIO', 'dc.identifier'), ('COLLOCAZIONE', 'dc.identifier'), ('FONDO', 'dc.relation.fonds'), ('TITOLO', "dc.title (Catalogo d'asta)"), ("NOME DELL'EVENTO", 'dc.title'), ('Unnamed: 6_level_0', 'Unnamed: 6_level_1'), ('Unnamed: 7_level_0', 'Unnamed: 7_level_1'), ('LUOGO', 'oairecerif.event.place'), ('BANDITORE', 'dc.contributor.auctioneer'), ('ORGANIZZATORE', 'dc.contributor.corporatebody'), ('ORGANIZZAZIONI COIVOLTE', 'dc.contributor.othercorporatebody'), ('PERSONE COINVOLTE', 'dc.contributor.contributor'), ('Unnamed: 13_level_0', 'Unnamed: 13_level_1'), ('CRONOLOGIA DEGLI OGGETTI IN VENDITA', 'dc.coverage.temporal'), ('TIPOLOGIA DEGLI OGGETTI IN VENDITA', 'dcterms.subject'), ('COLLEZIONI_IN VENDITA', 'dc.description.collectionauction'), ('0', 'Unnamed: 17_level_1'), ('Unnamed: 18_level_0', 'Unnamed: 18_level_1')]


# Prepare Graph

In [20]:
!pip install rdflib
import rdflib
from rdflib import Namespace, URIRef, Literal, Graph, ConjunctiveGraph, RDF, RDFS, XSD
from rdflib.store import Store

# Common namespaces
RDF = Namespace("http://www.w3.org/1999/02/22-rdf-syntax-ns#")
RDFS = Namespace("http://www.w3.org/2000/01/rdf-schema#")
XSD = Namespace("http://www.w3.org/2001/XMLSchema#")
DC = Namespace("http://purl.org/dc/elements/1.1/")
CRM = Namespace("http://www.cidoc-crm.org/cidoc-crm/")
LA = Namespace("https://linked.art/ns/terms/")
AAT = Namespace("http://vocab.getty.edu/aat/")

# Custom namespace
ZAC = Namespace("http://w3id.org/zac/")

# Create a context-aware graph using a Memory store
g = Graph()

# Bind namespaces to the graph
g.bind("rdf", RDF)
g.bind("rdfs", RDFS)
g.bind("xsd", XSD)
g.bind("dc", DC)
g.bind("zac", ZAC)
g.bind("crm", CRM)
g.bind("la", LA)
g.bind("aat", AAT)

print(df.columns)

Index(['INVENTARIO', 'COLLOCAZIONE', 'TITOLO', 'ALTRITITOLI', 'AUTORE',
       'BANDITORE', 'AUTORE SECONDARIO', 'ENTE AUTORE',
       'ENTE AUTORE SECONDARIO', 'DATA', 'LINGUA', 'SECOLO', 'PERIODO STORICO',
       'DIMENSIONI', 'LUOGO + EDITORE + DATA', 'FA PARTE DI', 'DESCRIZIONE2',
       'LUOGO', 'Catalogo Fondazione Zeri', 'Altri Collegamenti',
       'Tipo di supporto', 'Tipo di oggetto', 'Materiale e tecnica',
       'Vendita all'asta', 'Fondo', 'Progetto', 'Livello bibliografico',
       'Tipologia del materiale', 'Tipologia COAR', 'Provider',
       'Data provider', 'Licenza', 'Detentore dei diritti'],
      dtype='object')


# Utils

In [21]:
def create_uri_string(input_string):
    """
    Creates a URI-friendly string from an input string by replacing spaces
    with underscores and substituting special characters with similar
    non-special characters, and converting the result to lowercase.

    Args:
        input_string: The string to convert.

    Returns:
        A URI-friendly string in lowercase, or None if the input is None or NaN.
    """
    if pd.isna(input_string):
        return input_string.tostring()

    # Replace spaces with underscores
    uri_string = input_string.strip().replace(" ", "_")

    # Define a mapping for special characters to similar non-special characters
    char_replacements = {
        'à': 'a', 'è': 'e', 'é': 'e', 'ì': 'i', 'ò': 'o', 'ù': 'u',
        'À': 'A', 'È': 'E', 'É': 'E', 'Ì': 'I', 'Ò': 'O', 'Ù': 'U',
        "'": "", '"': "", "‘": "", "’": "",  # Remove quotes
        "(": "", ")": "", "[": "", "]": "", "{": "", "}": "", # Remove brackets
        ",": "", ";": "", ":": "", ".": "", "!": "", "?": "", # Remove punctuation
        "&": "and",  # Replace ampersand
        "/": "_", # Replace slash
        "\\": "_", # Replace backslash
    }

    # Apply character replacements
    for old_char, new_char in char_replacements.items():
        uri_string = uri_string.replace(old_char, new_char)

    # Remove any remaining characters that are not alphanumeric, underscores, or hyphens
    uri_string = re.sub(r'[^\w-]', '', uri_string)

    # Convert the entire string to lowercase
    uri_string = uri_string.lower()

    return uri_string

def extract_name_and_place(input_string):
    """
    Extracts a name and an optional place name from a string.

    Assumes the format is "Name <Place>" where <Place> is optional.

    Args:
        input_string: The input string.

    Returns:
        A tuple containing the name and the place name. The place name
        will be None if not present in the input string.
    """
    if pd.isna(input_string):
        return None, None

    # Regex to capture the name and the optional part in angle brackets
    match = re.match(r"([^<]+)(?:\s*<([^>]+)>)?", input_string.strip())

    if match:
        name = match.group(1).strip()
        place = match.group(2).strip() if match.group(2) else None
        return name, place
    else:
        # Return the original string as name if no match, and None for place
        return input_string.strip(), None

def role_assignment(uri_catalogue_creation, string_agent, uri_role):
  g.add((URIRef(uri_catalogue_creation+'_assignment_'+create_uri_string(string_agent)), RDF.type, CRM.E13_Attribute_Assignment))
  g.add((URIRef(uri_catalogue_creation+'_assignment_'+create_uri_string(string_agent)), CRM.P140_assigned_attribute_to, URIRef(uri_catalogue_creation) ))
  g.add((URIRef(uri_catalogue_creation+'_assignment_'+create_uri_string(string_agent)), CRM.P141_assigned, URIRef(ZAC[create_uri_string(string_agent)] ) ))
  g.add((URIRef(uri_catalogue_creation+'_assignment_'+create_uri_string(string_agent)), CRM.P177_assigned_property_type, CRM.P14_carried_out_by ))
  g.add((URIRef(uri_catalogue_creation+'_assignment_'+create_uri_string(string_agent)), CRM.P2_has_type, URIRef(uri_role) ))

def create_short_id(text, length=12):
    return hashlib.sha256(text.encode()).hexdigest()[:length]

def parse_date_value(date_val, default_xsd_val=None):
    """
    Parses a date value, which can be an integer (YYYYMMDD), float, or string.
    Returns a tuple: (formatted_label_string, xsd_literal_date).
    Handles 'nd' for end dates by optionally using the start date's XSD literal.
    """
    formatted_label = ""
    xsd_literal = None

    if pd.isna(date_val):
        return None, None

    date_str_raw = str(date_val).strip()

    # Handle 'nd' specifically for end dates
    if date_str_raw.lower() == "nd":
        return date_str_raw, default_xsd_val

    # Attempt to convert to a numeric string (e.g., '19070304' from 19070304.0)
    numeric_date_str = None
    try:
        # If it's a number, convert to int to remove trailing .0
        if date_str_raw.replace('.', '', 1).isdigit():
            numeric_date_str = str(int(float(date_str_raw)))
        else:
            # Not purely numeric, try to parse as is, or fall back to original string
            numeric_date_str = date_str_raw
            print("numeric_date_str",numeric_date_str)
    except (ValueError, TypeError):
        # If conversion to float/int fails, it's definitely not YYYYMMDD numeric
        print("ValueError, TypeError",ValueError, TypeError)
        pass

    if numeric_date_str and len(numeric_date_str) == 8: # Assuming YYYYMMDD
        try:
            parsed_date = pd.to_datetime(numeric_date_str, format='%Y%m%d', errors='coerce')
            if pd.notna(parsed_date):
                xsd_literal = Literal(parsed_date.strftime('%Y-%m-%d'), datatype=XSD.date)
                formatted_label = parsed_date.strftime('%Y/%m/%d')
            else:
                # Failed to parse YYYYMMDD, perhaps an invalid day/month
                formatted_label = date_str_raw[:4]
                xsd_literal = Literal(formatted_label, datatype=XSD.gYear)
                print("formatted_label",formatted_label)
        except ValueError:
            # Should be caught by errors='coerce', but as a fallback
            formatted_label = date_str_raw
            print("ValueError formatted_label",formatted_label,ValueError)
    elif numeric_date_str and len(numeric_date_str) == 6: # Assuming YYYYMM
        # This is where '1928-03-' might come from. We can't form a full date.
        # So, we just represent it as YYYY/MM for the label, and leave XSD as None.
        formatted_label = f"{numeric_date_str[0:4]}/{numeric_date_str[4:6]}"
        xsd_literal = Literal(formatted_label.replace("/","-"), datatype=XSD.gMonth)
        print("formatted_label 6",formatted_label)
    else:
        formatted_label = date_str_raw # Fallback for non-numeric or other formats
        print("formatted_label else",formatted_label)

    return formatted_label, xsd_literal

# Dictionaries

## Cronologia oggetti venduti

In [22]:
# Get unique values from the "CRONOLOGIA DEGLI OGGETTI IN VENDITA" column, dropping NaN values
unique_cronologia_values = df_asta[('CRONOLOGIA DEGLI OGGETTI IN VENDITA', 'dc.coverage.temporal')].dropna().unique()

# Create a dictionary with unique values as keys and empty strings as values
cronologia_dict = {value: "" for value in unique_cronologia_values}

aat_values = {
    "IV": "300404496",
    "V": "",
    "VI": "",
    "VII": "",
    "VIII": "",
    "IX": "",
    "X": "300404502",
    "XI": "300404503",
    "XII": "300404504",
    "XIII": "300404505",
    "XIV": "300404506",
    "XV": "300404465",
    "XVI": "300404510",
    "XVII": "300404511",
    "XVIII": "300404512",
    "XIX": "300404513",
    "XX": "300404514",
}

timespan_objects = {}

century_roman_numerals_ordered = [
    "IV", "V", "VI", "VII", "VIII", "IX", "X",
    "XI", "XII", "XIII", "XIV", "XV", "XVI", "XVII", "XVIII", "XIX", "XX"
]

# Regex to find Roman numerals for centuries IV-XX
roman_numeral_century_regex = r'\b(?:IV|V|VI|VII|VIII|IX|X|XI|XII|XIII|XIV|XV|XVI|XVII|XVIII|XIX|XX)\b'

for cronologia_key in cronologia_dict.keys():
    explicit_roman_numerals = re.findall(roman_numeral_century_regex, cronologia_key, re.IGNORECASE)
    explicit_roman_numerals = [rn.upper() for rn in explicit_roman_numerals]

    centuries_to_include = set()

    if len(explicit_roman_numerals) >= 2:
        # Assume a range, using the first and last found Roman numerals as bounds
        # Sort them to ensure correct range calculation if they are out of order in the string
        sorted_explicit_rns = sorted(explicit_roman_numerals, key=lambda x: century_roman_numerals_ordered.index(x) if x in century_roman_numerals_ordered else -1)

        if len(sorted_explicit_rns) >= 2 and all(rn in century_roman_numerals_ordered for rn in [sorted_explicit_rns[0], sorted_explicit_rns[-1]]):
            start_rn = sorted_explicit_rns[0]
            end_rn = sorted_explicit_rns[-1]

            start_index = century_roman_numerals_ordered.index(start_rn)
            end_index = century_roman_numerals_ordered.index(end_rn)

            # Include all centuries within the range (inclusive)
            for i in range(start_index, end_index + 1):
                centuries_to_include.add(century_roman_numerals_ordered[i])
        else:
            # If not a valid range in our ordered list, just add the explicit ones
            for rn in explicit_roman_numerals:
                centuries_to_include.add(rn)

    elif len(explicit_roman_numerals) == 1:
        # Single century, e.g., "Sec. XVI"
        centuries_to_include.add(explicit_roman_numerals[0])
    # If no Roman numerals are found, centuries_to_include remains empty

    aat_ids_for_key = set()
    for rn in centuries_to_include:
        if rn in aat_values and aat_values[rn]: # Check if Roman numeral exists in aat_values and has a non-empty ID
            aat_ids_for_key.add(aat_values[rn]) # Add AAT ID string

    timespan_objects[cronologia_key] = sorted(list(aat_ids_for_key))

print("Cronologia to AAT ID mapping (including intermediate centuries):")
display(timespan_objects)

Cronologia to AAT ID mapping (including intermediate centuries):


{'Sec. XV/ XVII': ['300404465', '300404510', '300404511'],
 'Sec. XV/ XVI': ['300404465', '300404510'],
 'Sec. XIV/ XVIII': ['300404465',
  '300404506',
  '300404510',
  '300404511',
  '300404512'],
 'Sec. XIV/ XVI': ['300404465', '300404506', '300404510'],
 'Sec. XIV/ XIX': ['300404465',
  '300404506',
  '300404510',
  '300404511',
  '300404512',
  '300404513'],
 'Sec. XV/ XVIII': ['300404465', '300404510', '300404511', '300404512'],
 'Sec. XV/ XIX': ['300404465',
  '300404510',
  '300404511',
  '300404512',
  '300404513'],
 'Sec. XVI/ XIX': ['300404510', '300404511', '300404512', '300404513'],
 'Sec. XVI/ XVIII': ['300404510', '300404511', '300404512'],
 'Sec. XVIII': ['300404512'],
 'Sec. XVII/ XVIII': ['300404511', '300404512'],
 'Sec. XVI/ XVII': ['300404510', '300404511'],
 'Sec. XVI': ['300404510'],
 'Sec. XVI/ XX': ['300404510',
  '300404511',
  '300404512',
  '300404513',
  '300404514'],
 'Sec. XV': ['300404465'],
 'Sec. XVIII/ XVIII': ['300404512'],
 'Sec. XIX': ['300404513']

## Roles (lost in the second version)

Mostly "mercante d'arte" (200 occurrences)

## Object types

In [23]:
object_types = {
 'DIPINTI': '300033618',
 "OGGETTI D'ARTE": '300133005',
 'SCULTURE': '300047090',
 'DISEGNI': '300033973',
 'LIBRI': '300028051',
 'TESSUTI': '300231565',
 'MOBILI': '300037680',
 'MEDAGLIE': '300046025',
 'VETRI': '300206140',
 'ARMI': '300036926',
 'STAMPE': '300041273',
 'CERAMICHE': '300010666',
 'MINIATURE': '300404587',
 'ARAZZI': '300205002',
 'MAIOLICHE': '300021170',
 'STRUMENTI MUSICALI': '300041620',
 'DIPNTI': '300033618',
 'SMALTI': '300178264',
 'AVORII': '300047325',
 'REPERTI ARCHEOLOGICI': '300234110',
 'MONETE': '300037222',
 'ARGENTI': '300234016',
 'GIOIELLI': '300209286',
 'ABITI': '300266639',
 'STOFFE': '300231565',
 'OROLOGI': '300041615',
 'ARCHEOLOGIA': '300234110',
 '': '',
 "GGETTI D'ARTE": '300133005',
 'ARGENTERIE': '300234016',
 'DIPINTI Sec. 19.': '300033618',
 'MANOSCRITTI': '300265483',
 'LENOBEL': '',
 'SCUTURE': '300047090',
 'STRUMENTI SCIENTIFICI': '300122283',
 'DIESEGNI': '300033973',
 'DIPINTI Sec. 18.': '300033618',
 'CERARMICHE': '300010666',
 'LETTERE': '300026879',
 'FERRO BATTUTO': '300011012',
 'VINI': '300379442',
 'DIPINIT': '300033618',
 'INCISIONI': '300041340',
 'TAPPETI': '300185756',
 'AVORI': '300047325',
 'DISRGNI': '300033973',
 'VETRI ISTORIATI': '300206140',
 'PITTURA': '300033618',
 "0GGETTI D'ARTE": '300133005',
 'QUADRI': '300404391',
 'VASI': '300132254',
 'PORCELLANE': '300010662',
 'MAIOLICHE DI DELFT': '300021170',
 'DIPINTI Sec. 19.-20.': '300033618',
 'DISEGNI Sec. 19.-20.': '300033973',
 'DIPINTI Sec. 18.-20.': '300033618',
 'DISEGNI Sec. 18.-20.': '300033973',
 'DIPINTI Sec. 17.-18.': '300033618',
 'DISEGNI Sec. 16.-19.': '300033973',
 'DIPINTI Sec. 15.-16.': '300033618',
 'SCULTURE Sec. 13.-16.': '300047090',
 'PISANELLO': '',
 'MINIATURE Sec. 13.-15.': '300404587',
 'BLEIBINHAUS A.': '',
 "OGGETTI D'ARTE Sec. 18.-20.": '300133005',
 'DIPINTI Sec. 14.-17.': '300033618',
 'DIPINTI Sec. 16.-19.': '300033618',
 'ARMATURE': '300036745',
 'BRONZI Sec. 14.-18.': '300047333',
 'SCHNEIDER & HANAU': '',
 'PITTURA Sec. 19.-20.': '300033618',
 'LIPHART, Karl Eduard Freiherr von': '',
 'SCHEFIK PASCHA': '',
 'HORST, WILTH': '',
 "OGGETTI D'ARTE Sec. 14-16.": '300133005',
 'ERGAS, RUDOLF': '',
 "OGGETTI D'ARTE Sec. 14.-16.": '300133005',
 'SCHLÖSSER, KARL': '',
 'WOLFF, AUGUST': '',
 "OGGETTI D'ARTE Sec. 16.-18.": '300133005',
 'JACOB DOPPLER': '',
 'DIPINTI Sec. 15.-19.': '300033618',
 'SCULTURA Sec. 19.-20.': '300047090',
 'DIPINTI NAPOLETANI': '300033618',
 'ISENBURG, KARL VON': '',
 'ROTHSCHILD, D.': '',
 'GOEDECKER, CARL': '',
 'KÖSTER': '',
 'SULZBACH, EMIL': '',
 'DIPINTI Sec. 16.-18.': '300033618',
 'DIPINTI Sec. 16.-20.': '300033618',
 'CORINTH, LOVIS': '300033618',
 'PITTURA Sec. 16.-19.': '300033618',
 'BAYERN, GISELA von': '',
 'ACQUERELLI Sec. 19.-20.': '300078925',
 'DIPINTI Sec. 20.': '300033618',
 'DISEGNI Sec. 20.': '300033973',
 'ARREDAMENTO Sec. 20.': '300037680',
 'LÖWITH, WILHELM': '',
 "OGGETTI D'ARTE Sec. 15.-18.": '300133005',
 'SPARR': '',
 'NEMES, MARCEL von': '',
 'DEYM': '',
 'HOHENTHAL': '',
 'LIPPERHEIDE, ELISABETH': '',
 'MAIOLICHE ITALIANE Sec. 15.-16.': '300021170',
 'STRAUSS, OTTMAR': '',
 'ACQUARELLI Sec. 19.-20.': '300078925',
 'PITTURA Sec. 19.-20': '300033618',
 'ZIETHEN, FELIX': '',
 'PITTURA Sec. 16.-18.': '300033618',
 'BOLIN': '',
 'LANDAU': '',
 'HEILAND': '',
 'PORCELLANE DI DOCCIA Sec. 18.': '300010662',
 'EISENMANN': '',
 'SCHÜSSLER': '',
 'SCHWARZ': '',
 'Altkunst Antiquitäten': '',
 'Galeria van Diemen & Co': '',
 "OPERE D'ARTE": '300133005',
 "COLLEZIOBNE D'HEUCQUEVILLE": '',
 'FRANCISCO GOYA': '',
 'BUDGE, EMMA': '',
 'ARREDAMENTO': '300037680',
 'SCULTURE IN LEGNO Sec. 14.-18.': '300047090'}

# Get unique values from the 'TIPOLOGIA DEGLI OGGETTI IN VENDITA' column
unique_object_types = df_asta[('TIPOLOGIA DEGLI OGGETTI IN VENDITA', 'dcterms.subject')].dropna().unique()

# Create a mapping dictionary
object_type_mapping = {}
for obj_type_key in unique_object_types:
    # Split the key by ';' to handle multiple object types in one cell
    individual_types = [t.strip() for t in obj_type_key.split(';')]

    aat_ids_for_key = set() # Use a set to store unique AAT IDs for the current obj_type_key

    for individual_type in individual_types:
        # Clean the individual type to match dictionary entries, e.g., strip spaces and convert to uppercase
        cleaned_individual_type = individual_type.strip().upper()

        # Attempt to find a direct match first
        if cleaned_individual_type in object_types:
            if object_types[cleaned_individual_type]: # Only add if it has a non-empty AAT ID
                aat_ids_for_key.add(object_types[cleaned_individual_type])
        else:
            # For entries like 'DIPINTI Sec. 19.', try to extract the base type
            base_type_match = re.match(r'([A-Za-z\s]+)(?: Sec\.\s[0-9.-]+)?', cleaned_individual_type)
            if base_type_match:
                base_type = base_type_match.group(1).strip()
                if base_type in object_types and object_types[base_type]:
                    aat_ids_for_key.add(object_types[base_type])

    object_type_mapping[obj_type_key] = sorted(list(aat_ids_for_key)) if aat_ids_for_key else [''] # Return sorted list or empty string if no IDs found

print("Object Type to AAT ID mapping:")
display(object_type_mapping)


Object Type to AAT ID mapping:


{'DIPINTI': ['300033618'],
 "OGGETTI D'ARTE; DIPINTI": ['300033618', '300133005'],
 "OGGETTI D'ARTE": ['300133005'],
 "OGGETTI D'ARTE; SCULTURE; DIPINTI; DISEGNI; AVORI": ['300033618',
  '300033973',
  '300047090',
  '300047325',
  '300133005'],
 "OGGETTI D'ARTE; DIPINTI; LIBRI": ['300028051', '300033618', '300133005'],
 "OGGETTI D'ARTE; TESSUTI; DISEGNI; DIPINTI": ['300033618',
  '300033973',
  '300133005',
  '300231565'],
 "OGGETTI D'ARTE; DIPINTI; DISEGNI; SCULTURE; MAIOLICHE": ['300021170',
  '300033618',
  '300033973',
  '300047090',
  '300133005'],
 "OGGETTI D'ARTE; TESSUTI; DIPINTI": ['300033618', '300133005', '300231565'],
 "OGGETTI D'ARTE; MOBILI; MEDAGLIE; VETRI; ARMI": ['300036926',
  '300037680',
  '300046025',
  '300133005',
  '300206140'],
 "OGGETTI D'ARTE; STAMPE; MOBILI; SCULTURE; VETRI; CERAMICHE; DISEGNI; DIPINTI": ['300010666',
  '300033618',
  '300033973',
  '300037680',
  '300041273',
  '300047090',
  '300133005',
  '300206140'],
 "OGGETTI D'ARTE; CERAMICHE; MAIOLI

## Language

In [24]:
langs = {"German":"de", "French":"fr","Italian":"it", "English":"en", "abs": "it"}
langs_labels_it = {"German":"Tedesco", "French":"Francese","Italian":"Italiano", "English":"Inglese", "abs": "Italiano"}

## Library sections

In [25]:
library_sections = {
    "CA 01": "Inghilterra",
    "CA 02": "Svezia",
    "CA 03": "Olanda",
    "CA 04": "Belgio",
    "CA 05": "Grecia-Ungheria",
    "CA 06": "Grecia-Ungheria",
    "CA 07": "Grecia-Ungheria",
    "CA 08": "Grecia-Ungheria",
    "CA 09": "Grecia-Ungheria",
    "CA 10": "Grecia-Ungheria",
    "CA 11": "Grecia-Ungheria",
    "CA 12": "Svizzera",
    "CA 13": "America",
    "CA 14": "Austria",
    "CA 15": "Danimarca",
    "CA 16": "Italia",
    "CA 17": "Francia",
    "CA 18": "Germania",
    "CA 19": "Arte moderna",
    "CC 19": "Arte moderna",
    "CA 20": "Fiere antiquariato",
    "CA 21": "Tappeti, arazzi",
    "CA 22": "Mobili",
    "CA 23": "Cornici",
    "CA 24": "Vetri",
    "CA 25": "Ceramica",
    "CA 26": "Porcellana",
    "CA 27": "Maiolica",
    "CA 28": "Argenti",
    "CA 29": "Auto boxes",
    "CA 30": "Auto boxes",
    "CA 31": "Auto boxes",
    "CA 32": "Auto boxes",
    "CA 33": "Gioielli",
    "CA 34": "Orologi, strumenti scientifici",
    "CA 35": "Paperweights",
    "CA 36": "Strumenti musicali",
    "CA 37": "Vini Posateria",
    "CA 38": "Vini Posateria",
    "CA 61": "Collezioni",
}

In [26]:
# extract which library sections are actually included
five_char_prefixes_from_collocazione = list()
# Rimosso .iloc[:, 0] perché df['COLLOCAZIONE'] è ora una Series (1D)
collocazione_list = df['COLLOCAZIONE'].dropna().to_list()

for collocazione_value in collocazione_list:
  if isinstance(collocazione_value, str):
    parts = collocazione_value.split(';')
    for part in parts:
        cleaned_part = part.strip()
        if len(cleaned_part) >= 5:
            five_char_prefixes_from_collocazione.append(cleaned_part[:5])
        elif cleaned_part: # Add if not empty but less than 5 characters
            five_char_prefixes_from_collocazione.append(cleaned_part)

five_char_prefixes_from_collocazione = sorted(list(set(five_char_prefixes_from_collocazione)))
print(five_char_prefixes_from_collocazione)
# ['CA 01', 'CA 03', 'CA 04', 'CA 12', 'CA 13', 'CA 16', 'CA 17', 'CA 18', 'CC 19']

['CA 01', 'CA 03', 'CA 04', 'CA 12', 'CA 13', 'CA 16', 'CA 17', 'CA 18', 'CC 19']


## Publication place

In [27]:
edition_list = df['LUOGO + EDITORE + DATA'].dropna().to_list()

places = set()
for edition_value in edition_list:
  if "s.l" in edition_value.lower() or "sl." in edition_value.lower():
    continue

  elif ":" in edition_value:
    place = edition_value.split(":")[0].replace("[","").replace("]","").replace("(","").strip()
    if place == "s.d.":
      continue
    places.add(place)

# 1. pulizia base
def clean(s):
    s = s.strip().strip(';').strip()
    s = re.sub(r'\s+', ' ', s)
    return s

cleaned = {p: clean(p) for p in places}

# 2. mappa manuale per traduzioni / nomi noti
translations = {
    'amburgo': 'Hamburg', 'hamburg': 'Hamburg',
    'münchen': 'München', 'munchen': 'München', 'müunchen': 'München',
    'monaco': 'München', 'monaco di baviera': 'München', 'münich': 'München',
    'munich': 'München', 'munchen & leipzig': 'München',
    'köln': 'Köln', 'koln': 'Köln', 'coln': 'Köln', 'cöln': 'Köln', 'colonia': 'Köln',
    'cologne': 'Köln', 'bonn-köln': 'Köln',
    'firenze': 'Firenze', 'florence': 'Firenze',
    'napoli': 'Napoli',
    'roma': 'Roma', 'rome': 'Roma', 'berlino': 'Berlin', 'berlin': 'Berlin',
    'milano': 'Milano', 'milan': 'Milano',
    'venedig': 'Venezia', 'venise': 'Venezia', 'venezia': 'Venezia',
    'perouse': 'Perugia',
    'francoforte sul meno': 'Frankfurt am Main', 'frankfurt. a. m.': 'Frankfurt am Main',
    'frankfurt': 'Frankfurt am Main', 'frankfurt a. m.': 'Frankfurt am Main',
    'frankfurt a.m.': 'Frankfurt am Main', 'frankfurt a. main': 'Frankfurt am Main',
    'frankfurt am main': 'Frankfurt am Main', 'frankfurt a/m': 'Frankfurt am Main',
    'frankurt a. m.': 'Frankfurt am Main', 'hanau am main': 'Hanau am Main',
    'bonn am rhein': 'Bonn', 'bonn a. rhein': 'Bonn', 'bonn': 'Bonn',
    'aachen': 'Aachen', 'aaachen': 'Aachen',
    'lübeck': 'Lübeck', 'lubeck': 'Lübeck',
    'stuttgart': 'Stuttgart', 'stuttgard': 'Stuttgart',
    's. l.': None,
}

def normalize(name):
    if not name: return None
    key = name.lower().strip()
    return translations.get(key, name)

# 3. Fuzzy clustering e mapping finale
canonical_names = set()
for p in places:
    norm = normalize(clean(p))
    if norm: canonical_names.add(norm)

canonical_list = list(canonical_names)
fuzzy_mapping = {}
for name in canonical_list:
    if name in fuzzy_mapping:
        continue
    matches = process.extract(name, canonical_list, scorer=fuzz.WRatio, limit=None)
    for match_name, score, _ in matches:
        if score >= 90:
            fuzzy_mapping.setdefault(name, name)
            fuzzy_mapping[match_name] = name

# Creazione del dizionario finale: { Originale: Pulito }
place_mapping_final = {}
for original in places:
    step1 = clean(original)
    step2 = normalize(step1)
    if step2:
        final_name = fuzzy_mapping.get(step2, step2)
        place_mapping_final[original] = final_name

print(f"Mappatura creata per {len(place_mapping_final)}/{len(places)} luoghi originali.")
# Esempio:
display(place_mapping_final)

Mappatura creata per 72/73 luoghi originali.


{'New York': 'New York',
 'Frankfurt a.m.': 'Frankfurt am Main',
 'Bonn-Köln': 'Köln',
 'Stuttgart': 'Stuttgart',
 'Rome': 'Roma',
 'Frankfurt a. Main': 'Frankfurt am Main',
 'Francoforte sul Meno': 'Frankfurt am Main',
 'Frankfurt a/M': 'Frankfurt am Main',
 '; München': 'München',
 'Perouse': 'Perugia',
 'Venezia': 'Venezia',
 'Monaco': 'München',
 'Cöln': 'Köln',
 'Amsterdam': 'Amsterdam',
 'Berlin': 'Berlin',
 'Koln': 'Köln',
 'Frankurt a. M.': 'Frankfurt am Main',
 'Hanau am Main': 'Hanau am Main',
 'Lubeck': 'Lübeck',
 'Munchen & Leipzig': 'München',
 'Frankfurt. a. M.': 'Frankfurt am Main',
 'Rees': 'Rees',
 'Hamburg': 'Hamburg',
 'Genova': 'Genova',
 'London': 'London',
 'Napoli': 'Napoli',
 'Roma': 'Roma',
 'Coln': 'Köln',
 'Colonia': 'Köln',
 'Leipzig': 'Leipzig',
 'Müunchen': 'München',
 'Berlino': 'Berlin',
 'Frankfurt am Main': 'Frankfurt am Main',
 'Cologne': 'Köln',
 'Weimar': 'Weimar',
 'Milano': 'Milano',
 'Freiburg im Breisgau': 'Freiburg im Breisgau',
 'Hellerau': 'H

## Publisher

In [28]:
# retrieve all publishers and save the list in a spreadsheet, then manually normalise them
publishers = set()
for edition_value in edition_list:
  if "s.n" in edition_value.lower():
    continue

  elif ":" in edition_value:
    publisher = edition_value.split(":")[1].split(",")[0].strip()
    publishers.add(publisher)

# Graph generation

## Changelog

Current version July 2026.

- REMOVED second row from spreadsheet "Zeri CATALOGHI" because of parsing issues
- ADDED: catalogue E31 -> P148 -> E33 Linguistic object
- ADDED: catalogue language associated to E33 above (not replacing previous pattern)
- ADDED: E31 -> P48 -> shelfmark identifier
- ADDED: E31 -> P70i -> Historica Handle page
- ADDED: E31 -> P138i -> Historica IIIF manifest
- ADDED: E31 -> P128i -> E22 physical object, in turn associated with:
  - inventory number (P48),
  - shelfmark (P48),
  - dimensions (P43)
  - production: in turn linked to
    - Publisher
    - publication place
    - date (same as creation date)
- ADDED: E31 -> P148i -> library section (extracted from shelfmark/COLLOCAZIONE)


Prev version
- lost people roles
- change patterns for inventory number (from literal to E42)
- added part of relations between catalogues and containers
- object types pattern changed:
  - AAT URI linked to auction directly with P125 + P2 "AAT_object_type",
  - local URI linked to auction directly with P125 + P2 "object_type"
- added codice asta
- added people involved, with CRM.P11_had_participant
- serialised in plain ttl
- added labels en/it
- added URL Zeri catalogue
- added full bibliographic reference of the catalogue
- added if illustrated or not

todo
- add shelf mark / collocazione as identifier (split and match with inventory number)

## Default triples

In [29]:
# MULTILINGUAL LABELS
g.add((URIRef(ZAC["illustrated"]), RDFS.label, Literal("Illustrato",lang="it") ))
g.add((URIRef(ZAC["illustrated"]), RDFS.label, Literal("Illustrated",lang="en") ))
g.add((URIRef(ZAC["primary_title"]), RDFS.label, Literal("Titolo",lang="it") ))
g.add((URIRef(ZAC["primary_title"]), RDFS.label, Literal("Title",lang="en") ))
g.add((AAT["300026068"], RDFS.label, Literal("Catalogo d'asta", lang="it")))
g.add((AAT["300026068"], RDFS.label, Literal("Auction catalogue", lang="en")))
g.add((AAT["300312355"], RDFS.label, Literal("Numero d'inventario", lang="it")))
g.add((AAT["300312355"], RDFS.label, Literal("Accession number", lang="en")))
g.add((AAT["300404704"], RDFS.label, Literal("Collocazione", lang="it")))
g.add((AAT["300404704"], RDFS.label, Literal("Shelfmark", lang="en")))
g.add((URIRef(ZAC["secondary_title"]), RDFS.label, Literal("Altro titolo",lang="it") ))
g.add((URIRef(ZAC["secondary_title"]), RDFS.label, Literal("Other title",lang="en") ))
g.add((AAT["300411307"], RDFS.label, Literal("Lotti", lang="it")))
g.add((AAT["300411307"], RDFS.label, Literal("Lots", lang="en")))
g.add((AAT["300025492"], RDFS.label, Literal("Autore", lang="it")))
g.add((AAT["300025492"], RDFS.label, Literal("Author", lang="en")))
g.add((ZAC["secondary_author"], RDFS.label, Literal("Autore secondario", lang="it")))
g.add((ZAC["secondary_author"], RDFS.label, Literal("Secondary author", lang="en")))
g.add((AAT["300054751"], RDFS.label, Literal("Asta", lang="it")))
g.add((AAT["300054751"], RDFS.label, Literal("Auction", lang="en")))
g.add(( URIRef(ZAC['aat_object_type']), RDFS.label, Literal("Tipologia oggetto (AAT)",lang="it") ))
g.add(( URIRef(ZAC['aat_object_type']), RDFS.label, Literal("AAT Object type",lang="en") ))
g.add(( URIRef(ZAC['object_type']), RDFS.label, Literal("Tipologia oggetto",lang="it") ))
g.add(( URIRef(ZAC['object_type']), RDFS.label, Literal("Object type",lang="en") ))
g.add(( URIRef(ZAC['auction_organisation']), RDFS.label, Literal("Organizzazione dell'asta",lang="it") ))
g.add(( URIRef(ZAC['auction_organisation']), RDFS.label, Literal("Auction organisation",lang="en") ))
g.add(( URIRef(ZAC['main_organiser']), RDFS.label, Literal("Organizzatore principale",lang="it") ))
g.add(( URIRef(ZAC['main_organiser']), RDFS.label, Literal("Main organiser",lang="en") ))
g.add(( URIRef(ZAC['secondary_organiser']), RDFS.label, Literal("Organizzatore secondario",lang="it") ))
g.add(( URIRef(ZAC['secondary_organiser']), RDFS.label, Literal("Other organiser",lang="en") ))
g.add(( URIRef(ZAC['auctioneering']), RDFS.label, Literal("Battitura",lang="it") ))
g.add(( URIRef(ZAC['auctioneering']), RDFS.label, Literal("Auctioneering",lang="en") ))
g.add((AAT["300025208"], RDFS.label, Literal("Battitore", lang="it") ))
g.add((AAT["300025208"], RDFS.label, Literal("Auctioneer", lang="en") ))
g.add((ZAC['person_involved'], RDFS.label, Literal("Altra persona coinvolta", lang="it")))
g.add((ZAC['person_involved'], RDFS.label, Literal("Person involved", lang="en")))
g.add(( URIRef(ZAC['codice_asta']), RDFS.label, Literal("Codice asta", lang="it")))
g.add(( URIRef(ZAC['codice_asta']), RDFS.label, Literal("Auction Identifier", lang="en")))

<Graph identifier=N91b39022135442d2baab54435889f8ba (<class 'rdflib.graph.Graph'>)>

## Catalogues

In [30]:
for index, row in df.iterrows():
    # CATALOGUE - Separate multiple catalogues
    val_id_cat = row['INVENTARIO']

    # Handle NaN values and ensure numeric IDs are treated as integers without '.0' suffix
    if pd.isna(val_id_cat):
        id_cat_str = "" # Treat NaN as an empty string for splitting purposes
    elif isinstance(val_id_cat, float) and val_id_cat.is_integer():
        id_cat_str = str(int(val_id_cat))
    else:
        id_cat_str = str(val_id_cat)

    # Separate multiple catalogues, ensuring empty strings are not processed
    id_cats = [create_uri_string(part.strip().replace(" ","")) for part in id_cat_str.split(';') if part.strip()]

    shelfmark = row['COLLOCAZIONE']
    title = row['TITOLO']
    secondary_title = row['ALTRITITOLI']
    date_cat = str(int(row['DATA'])).strip()
    lang_cat = langs[row['LINGUA']]
    lang_cat_label_it = langs_labels_it[row['LINGUA']]
    lang_cat_label_en = row['LINGUA']
    auction_name = row["Vendita all'asta"]
    author_person = row['AUTORE'] # TODO reconciliation person / place
    author_org = row['ENTE AUTORE'] # TODO reconciliation org / place
    secondary_author_person = row["AUTORE SECONDARIO"]
    secondary_author_org = row["ENTE AUTORE SECONDARIO"]
    part_of = row["FA PARTE DI"]
    pub_place_publisher_date = row['LUOGO + EDITORE + DATA']

    publisher = None
    if pd.notna(pub_place_publisher_date) and ":" in pub_place_publisher_date:
      publisher_raw = pub_place_publisher_date.split(":")[1].split(",")[0].strip()

      # Lookup in df_publishers mapping table
      matching_row = df_publishers[df_publishers['original'] == publisher_raw]
      publisher = matching_row.iloc[0]['clean'] if not matching_row.empty else publisher_raw


    pub_dimensione = row['DIMENSIONI']
    id = id_cats[0] if len(id_cats) > 0 else None
    if id is None:
      continue
    #CATALOGUE TYPE
    g.add((URIRef(ZAC[id]), RDF.type, CRM.E31_Document))
    g.add((URIRef(ZAC[id]), CRM.P2_has_type, AAT["300026068"]))
    g.add((URIRef(ZAC[id]), RDFS.label, Literal(title, lang=lang_cat)))
    if pd.notna(pub_dimensione) and "tav." in pub_dimensione:
      g.add((URIRef(ZAC[id]), CRM.P2_has_type, URIRef(ZAC['illustrated']) ))

    # BIBLIOGRAPHIC REFERENCE
    bibl_str = ""
    if pd.notna(author_person):
      bibl_str += author_person
    if pd.notna(author_org):
      bibl_str += author_org + '. '
    if pd.notna(title):
      bibl_str += title + ". "
    if pd.notna(part_of):
      bibl_str += "In " + part_of + ". "
    if pd.notna(pub_place_publisher_date):
      bibl_str += pub_place_publisher_date
    if pd.notna(pub_dimensione):
      bibl_str += " - " + pub_dimensione + "."
    g.add((URIRef(ZAC[id]), CRM.P3_has_note, Literal(bibl_str)))

    # INVENTORY NUMBER 300312355
    g.add((URIRef(ZAC[id]), CRM.P48_has_preferred_identifier, URIRef(ZAC[id+'_id']) ))
    g.add((URIRef(ZAC[id+'_id']), RDF.type, CRM.E42_Identifier ))
    g.add((URIRef(ZAC[id+'_id']), RDFS.label, Literal(id) ))
    g.add((URIRef(ZAC[id+'_id']), CRM.P190_has_symbolic_content, Literal(id) ))
    g.add((URIRef(ZAC[id+'_id']), CRM.P2_has_type, AAT["300312355"] ))
    # COLLOCAZIONE / SHELF MARK 300404704
    g.add((URIRef(ZAC[id]), CRM.P48_has_preferred_identifier, URIRef(ZAC[id+'_shelfmark']) ))
    g.add((URIRef(ZAC[id+'_shelfmark']), RDF.type, CRM.E42_Identifier ))
    if isinstance(shelfmark, str) and ";" in shelfmark:
      shelfmark_parts = [create_uri_string(part.strip().replace(" ","")) for part in shelfmark.split(';')]
      if id_cats.index(id) == 0:
        shelfmark = shelfmark_parts[0]
      else:
        shelfmark = shelfmark_parts[1]
    g.add((URIRef(ZAC[id+'_shelfmark']), RDFS.label, Literal(shelfmark) ))
    g.add((URIRef(ZAC[id+'_shelfmark']), CRM.P190_has_symbolic_content, Literal(shelfmark) ))
    g.add((URIRef(ZAC[id+'_shelfmark']), CRM.P2_has_type, AAT["300404704"] ))
    ## longer path
    g.add((URIRef(ZAC[id]), CRM.P128i_is_carried_by, URIRef(ZAC[id+'_item']) ))
    g.add((URIRef(ZAC[id+'_item']), RDF.type, CRM["E22_Human-Made_Object"] ))
    g.add((URIRef(ZAC[id]), CRM.P48_has_preferred_identifier, URIRef(ZAC[id+'_id']) ))
    g.add((URIRef(ZAC[id]), CRM.P48_has_preferred_identifier, URIRef(ZAC[id+'_shelfmark']) ))

    #TITLE
    g.add((URIRef(ZAC[id]), CRM.P102_has_title, URIRef(ZAC[id+'_title']) ))
    g.add((URIRef(ZAC[id+'_title']), RDFS.label, Literal(title, lang=lang_cat)))
    g.add((URIRef(ZAC[id+'_title']), CRM.P190_has_symbolic_content, Literal(title, lang=lang_cat) ))
    g.add((URIRef(ZAC[id+'_title']), CRM.P2_has_type, URIRef(ZAC["primary_title"])))
    #SECONDARY TITLE
    if pd.notna(secondary_title):
      g.add((URIRef(ZAC[id]), CRM.P102_has_title, URIRef(ZAC[id+'_secondary_title']) ))
      g.add((URIRef(ZAC[id+'_secondary_title']), RDFS.label, Literal(secondary_title, lang=lang_cat)))
      g.add((URIRef(ZAC[id+'_secondary_title']), CRM.P190_has_symbolic_content, Literal(secondary_title, lang=lang_cat)))
      g.add((URIRef(ZAC[id+'_secondary_title']), CRM.P2_has_type, URIRef(ZAC["secondary_title"])))

    # LANGUAGE
    g.add((URIRef(ZAC[id]), CRM.P72_has_language, URIRef(ZAC['lang_'+lang_cat]) ))
    g.add((URIRef(ZAC['lang_'+lang_cat]), RDF.type, CRM.E56_Language ))
    ## longer path
    g.add((URIRef(ZAC[id]), CRM.P148_has_component, URIRef(ZAC[id+'_text']) ))
    g.add((URIRef(ZAC[id+'_text']), RDF.type, CRM.E33_Linguistic_Object ))
    g.add((URIRef(ZAC[id+'_text']), CRM.P72_has_language, URIRef(ZAC['lang_'+lang_cat]) ))
    g.add((URIRef(ZAC['lang_'+lang_cat]), RDFS.label, Literal(lang_cat_label_it, lang="it") ))
    g.add((URIRef(ZAC['lang_'+lang_cat]), RDFS.label, Literal(lang_cat_label_en, lang="en") ))
    # DATE CREATION
    g.add((URIRef(ZAC[id]), CRM.P94i_was_created_by, URIRef(ZAC[id+'_creation']) ))
    g.add((URIRef(ZAC[id+'_creation']), CRM.P82_at_some_time_within, Literal(date_cat, datatype=XSD.gYear)))
    ## longer path
    g.add(( URIRef(ZAC[id+'_creation']), CRM["P4_has_time-span"], URIRef(ZAC["date_"+date_cat]) ))
    g.add(( URIRef(ZAC["date_"+date_cat]), CRM.P82a_begin_of_the_begin, Literal(date_cat, datatype=XSD.gYear) ))

    # AUCTION NAME
    g.add((URIRef(ZAC[id]), CRM.P70_documents, URIRef(ZAC[id+'_auction']) ))
    g.add((URIRef(ZAC[id+'_auction']), RDFS.label, Literal(auction_name) ))
    g.add((URIRef(ZAC[id+'_auction']), CRM.P16_used_specific_object, URIRef(ZAC[id+'_lots']) ))
    g.add((URIRef(ZAC[id+'_lots']), CRM.P2_has_type, AAT["300411307"] ))

    # AUTHORS
    if pd.notna(author_person):
      for author_person_part in author_person.split("||"):
        g.add((URIRef(ZAC[id]+'_creation'), CRM.P14_carried_out_by, URIRef(ZAC[create_uri_string(author_person_part.strip())]) ))
        g.add((URIRef(ZAC[create_uri_string(author_person_part.strip())]), RDFS.label, Literal(author_person_part.strip()) ))
        g.add((URIRef(ZAC[create_uri_string(author_person_part.strip())]), RDF.type, CRM.E21_Person ))
        role_assignment(ZAC[id+'_creation'], author_person_part.strip(), AAT["300025492"])
    if pd.notna(author_org):
      for author_org_part in author_org.split("||"):
        g.add((URIRef(ZAC[id+'_creation']), CRM.P14_carried_out_by, URIRef(ZAC[create_uri_string(author_org_part.strip())]) ))
        g.add((URIRef(ZAC[create_uri_string(author_org_part.strip())]), RDFS.label, Literal(author_org_part.strip()) ))
        g.add((URIRef(ZAC[create_uri_string(author_org_part.strip())]), RDF.type, CRM.E74_Group ))
        role_assignment(ZAC[id+'_creation'], author_org_part.strip(), AAT["300025492"])
    if pd.notna(secondary_author_org):
      for secondary_author_org_part in secondary_author_org.split("||"):
        g.add((URIRef(ZAC[id+'_creation']), CRM.P14_carried_out_by, URIRef(ZAC[create_uri_string(secondary_author_org_part.strip())]) ))
        g.add((URIRef(ZAC[create_uri_string(secondary_author_org_part.strip())]), RDFS.label, Literal(secondary_author_org_part.strip()) ))
        g.add((URIRef(ZAC[create_uri_string(secondary_author_org_part.strip())]), RDF.type, CRM.E74_Group ))
        role_assignment(ZAC[id+'_creation'], secondary_author_org_part.strip(), ZAC["secondary_author"])
    if pd.notna(secondary_author_person):
      for secondary_author_person_part in secondary_author_person.split("||"):
        g.add((URIRef(ZAC[id]), CRM.P14_carried_out_by, URIRef(ZAC[create_uri_string(secondary_author_person_part.strip())]) ))
        g.add((URIRef(ZAC[create_uri_string(secondary_author_person_part.strip())]), RDFS.label, Literal(secondary_author_person_part.strip()) ))
        g.add((URIRef(ZAC[create_uri_string(secondary_author_person_part.strip())]), RDF.type, CRM.E21_Person ))
        role_assignment(ZAC[id+'_creation'], secondary_author_person_part.strip(), ZAC["secondary_author"])
        g.add((ZAC["secondary_author"], RDFS.label, Literal("Autore secondario", lang="it")))
    # PART OF
    if pd.notna(part_of):
      g.add(( URIRef(ZAC[id]), CRM.P148i_is_component_of, URIRef(ZAC[create_short_id(part_of, 8)]) ))
      g.add(( URIRef(ZAC[create_short_id(part_of, 8)]), RDFS.label, Literal(part_of, lang=lang_cat) ))
      g.add(( URIRef(ZAC[create_short_id(part_of, 8)]), RDF.type, CRM.E31_Document ))

    # SEZIONE
    if isinstance(shelfmark, str) and shelfmark[:5] in library_sections:
      section_name = library_sections[shelfmark[:5]]
      section_uri = shelfmark[:5].replace(" ", "_")
      g.add((URIRef(ZAC[id]), CRM.P148i_is_component_of, URIRef(ZAC['section_'+section_uri]) ))
      g.add((URIRef(ZAC['section'+section_uri]), RDFS.label, Literal(section_name, lang="it") ))

    # CONSISTENZA E DIMENSIONI
    g.add((URIRef(ZAC[id+'_item']), CRM.P43_has_dimension, URIRef(ZAC[id+'_item_dimension']) ))
    g.add((URIRef(ZAC[id+'_item_dimension']), RDFS.label, Literal(pub_dimensione) ))

    # PUBBLICAZIONE
    g.add((URIRef(ZAC[id+'_item']), CRM.P108i_was_produced_by, URIRef(ZAC[id+'_item_publication']) ))
    g.add((URIRef(ZAC[id+'_item_publication']), RDFS.label, Literal(pub_place_publisher_date) ))
    g.add((URIRef(ZAC[id+'_item_publication']), CRM.P2_has_type, AAT["300195853"] ))

    # LUOGO DI PUBBLICAZIONE
    if pd.notna(pub_place_publisher_date):
      for original_place, cleaned_place in place_mapping_final.items():
        if original_place in pub_place_publisher_date:
          g.add((URIRef(ZAC[id+'_item_publication']), CRM.P7_took_place_at, URIRef(ZAC[create_uri_string(cleaned_place)]) ))
          g.add((URIRef(ZAC[create_uri_string(cleaned_place)]), RDFS.label, Literal(cleaned_place) ))
          g.add((URIRef(ZAC[create_uri_string(cleaned_place)]), RDF.type, CRM.E53_Place ))
    # DATA DI PUBBLICAZIONE
    g.add(( URIRef(ZAC[id+'_item_publication']), CRM["P4_has_time-span"], URIRef(ZAC["date_"+date_cat]) ))
    # EDITORE
    if pd.notna(publisher):
        g.add((URIRef(ZAC[id+'_item_publication']), CRM.P14_carried_out_by, URIRef(ZAC[create_uri_string(publisher)]) ))
        g.add((URIRef(ZAC[create_uri_string(publisher)]), RDFS.label, Literal(publisher) ))
        g.add((URIRef(ZAC[create_uri_string(publisher)]), RDF.type, CRM.E74_Group ))
        role_assignment(ZAC[id+'_item_publication'], publisher, AAT["300025574"])

#g.serialize('zac_catalogues.ttl')

3 -  83625
3 -  83626; B 15101
3 -  87357
3 -  81742
3 -  83627
3 -  81743
3 -  5287
3 -  87358
3 -  81744
3 -  5731
3 -  87361
3 -  87362
3 -  81745
3 -  87363
3 -  87366
3 -  87365
3 -  87370; 87369
3 -  87371
3 -  87375
3 -  87374
3 -  87380
3 -  87379
3 -  87377
3 -  87378
3 -  81746
3 -  87389
3 -  87390
3 -  87393
3 -  87384
3 -  87381; 87382
3 -  87388
3 -  87394
3 -  87395
3 -  87385
3 -  87386
3 -  87396; 87397
3 -  87405
3 -  87408
3 -  87407
3 -  87404
3 -  87406
3 -  87403
3 -  87410
3 -  87399
3 -  87400
3 -  87401
3 -  87412
3 -  87411
3 -  83629
3 -  83628
3 -  87414
3 -  87415
3 -  87418
3 -  87417
3 -  83630
3 -  87424
3 -  87423
3 -  87421
3 -  87420
3 -  87419
3 -  87429
3 -  87428
3 -  87427
3 -  87426
3 -  87425
3 -  4466
3 -  81747
3 -  83631
3 -  83633
3 -  87436
3 -  87437
3 -  87438
3 -  87440
3 -  87432
3 -  81748
3 -  81749
3 -  81750
3 -  B 1210
3 -  83632
3 -  87450
3 -  87445
3 -  87446
3 -  87451
3 -  87452
3 -  87453
3 -  87455
3 -  87456
3 -  87457
3 - 

## IIIF Manifests

For catalogues manifests only, not single pages.

In [31]:
# Integration of IIIF Manifests from df_manifest into graph g
manifest_count = 0
seen_inv = set()

for index, row in df_manifest.iterrows():
    inv_val = str(row['inventario']) if pd.notna(row['inventario']) else ""
    handle_val = str(row['handle']) if pd.notna(row['handle']) else ""
    man_val = str(row['manifest']) if pd.notna(row['manifest']) else ""

    # Extract numeric part from 'BO0624_XXXXX'
    inv_match = re.search(r"BO0624_(\d+)", inv_val)
    # Validate manifest URL format
    man_match = re.search(r"https://historica\.unibo\.it/server/iiif/[0-9a-fA-F-]+/manifest", man_val)

    if inv_match and man_match:
        num = inv_match.group(1)
        manifest_url = man_match.group(0)

        # Add to graph if not already processed for this inventory number
        if num not in seen_inv:
            seen_inv.add(num)
            # Adding to the existing global graph g
            g.add((URIRef(ZAC[num]), CRM.P138i_has_representation, URIRef(manifest_url)))
            g.add((URIRef(ZAC[num]), CRM.P70i_is_documented_in, URIRef(handle_val) ))
            manifest_count += 1

print(f"Manifest mapping complete: added {manifest_count} triples to the graph.")
# Serialize again to include new triples
#g.serialize('zac_catalogues.ttl')

Manifest mapping complete: added 1891 triples to the graph.


## Auctions

In [32]:
# FROM AUCTION SPREADSHEET
for index, row in df_asta.iterrows():
  # CATALOGUE - Separate multiple catalogues
  id_cat = row[('INVENTARIO', 'dc.identifier')]
  id_cats = [create_uri_string(id.strip().replace(" ","")) if ';' in id_cat else create_uri_string(id_cat) for id in id_cat.split(';')]
  id_zeri_cat = row[('ID_CATALOGO Gestionale Zeri', 'Unnamed: 0_level_1')]

  auction_place = row[('LUOGO', 'oairecerif.event.place')] # TODO reconciliation
  auction_date_start = row[('Unnamed: 6_level_0', 'Unnamed: 6_level_1')]
  auction_date_end = row[('Unnamed: 7_level_0', 'Unnamed: 7_level_1')]
  auction_date_label = ""
  xsd_auction_date_start, xsd_auction_date_end = None, None

  # Parse start date
  start_label_part, xsd_auction_date_start = parse_date_value(auction_date_start)
  if start_label_part:
      auction_date_label += start_label_part

  # Parse end date, passing the start date's XSD literal for 'nd' handling
  end_label_part, xsd_auction_date_end_candidate = parse_date_value(auction_date_end, default_xsd_val=xsd_auction_date_start)

  if end_label_part:
      if auction_date_label: # If there's a start date label, add a separator
          auction_date_label += ' - '
      auction_date_label += end_label_part
      xsd_auction_date_end = xsd_auction_date_end_candidate # Update XSD end date

  # If auction_date_end was 'nd' but auction_date_start was None, xsd_auction_date_end will still be None
  # Ensure xsd_auction_date_end is at least xsd_auction_date_start if end was 'nd' and start was valid
  if pd.notna(auction_date_end) and str(auction_date_end).strip().lower() == "nd" and xsd_auction_date_start is not None:
      xsd_auction_date_end = xsd_auction_date_start

  object_types_value = row[('TIPOLOGIA DEGLI OGGETTI IN VENDITA', 'dcterms.subject')]
  object_period = row[('CRONOLOGIA DEGLI OGGETTI IN VENDITA', 'dc.coverage.temporal')]
  collections = row[('COLLEZIONI_IN VENDITA', 'dc.description.collectionauction')]
  collections_list = [id.strip() for id in str(collections).split(';')] if pd.notna(collections) else []
  organiser = row[('ORGANIZZATORE', 'dc.contributor.corporatebody')] # TODO reconciliation
  secondary_organiser = row[('ORGANIZZAZIONI COIVOLTE', 'dc.contributor.othercorporatebody')] # TODO reconciliation
  battitore= row[('BANDITORE', 'dc.contributor.auctioneer')] # TODO reconciliation
  battitori = [id.strip() for id in str(battitore).split('||')] if pd.notna(battitore) else []

  codice_asta = row[('Unnamed: 13_level_0', 'Unnamed: 13_level_1')] #NEW
  people_involved = row[('PERSONE COINVOLTE', 'dc.contributor.contributor')] #NEW
  people_involved_list = [id.strip() for id in str(people_involved).split('||')] if pd.notna(people_involved) else []

  id = id_cats[0]
  # ONLINE ZERI RECORD
  if pd.notna(id_zeri_cat):
    id_zeri_cat = str(int(id_zeri_cat))
    g.add((URIRef(ZAC[id]), CRM.P70i_is_documented_in, URIRef(ZAC[id_zeri_cat+"_record"]) ))
    g.add((URIRef(ZAC[id+'_auction']), CRM.P70i_is_documented_in, URIRef(ZAC[id_zeri_cat+"_record"]) ))
    g.add((URIRef(ZAC[id_zeri_cat+"_record"]), RDF.type, CRM.E31_Document ))
    g.add((URIRef(ZAC[id_zeri_cat+"_record"]), CRM.P1_is_identified_by, URIRef(ZAC[id_zeri_cat+'_url']) ))
    g.add((URIRef(ZAC[id_zeri_cat+'_url']), RDF.type, CRM.E42_Identifier ))
    g.add((URIRef(ZAC[id_zeri_cat+'_url']), CRM.P190_has_symbolic_content, Literal("https://catalogo.fondazionezeri.unibo.it/scheda/catalogodasta/"+id_zeri_cat+"/")))

  # AUCTION TYPE
  g.add(( URIRef(ZAC[id+'_auction']), RDF.type, CRM.E7_Activity )) # title and relation to catalogue already represented
  g.add(( URIRef(ZAC[id+'_auction']), CRM.P2_has_type, AAT["300054751"] ))

  # PLACE
  if pd.notna(auction_place):
    g.add(( URIRef(ZAC[id+'_auction']), CRM.P7_took_place_at, URIRef(ZAC[create_uri_string(auction_place)]) ))
    g.add(( URIRef(ZAC[create_uri_string(auction_place)]), RDFS.label, Literal(auction_place, lang="it") ))
    g.add(( URIRef(ZAC[create_uri_string(auction_place)]), RDF.type, CRM.E53_Place ))
  # DATE
  auction_date_uri_string = create_uri_string(auction_date_label.strip().replace(' - ', '-'))
  g.add(( URIRef(ZAC[id+'_auction']), CRM["P4_has_time-span"], URIRef(ZAC[auction_date_uri_string]) ))
  g.add(( URIRef(ZAC[auction_date_uri_string]), RDF.type, CRM["E52_Time-Span"] ))
  g.add(( URIRef(ZAC[auction_date_uri_string]), RDFS.label, Literal(auction_date_label) ))

  if xsd_auction_date_start:
    g.add(( URIRef(ZAC[auction_date_uri_string]), CRM.P82a_begin_of_the_begin, xsd_auction_date_start ))
  if xsd_auction_date_end:
    g.add(( URIRef(ZAC[auction_date_uri_string]), CRM.P82b_end_of_the_end, xsd_auction_date_end ))
  # TYPE OF OBJECTS
  if pd.notna(object_types_value):
    # Use the pre-computed object_type_mapping
    mapped_aat_ids = object_type_mapping.get(object_types_value, [])
    for aat_id in mapped_aat_ids:
      if aat_id:
        g.add(( URIRef(ZAC[id+'_auction']), CRM.P125_used_objects_of_type, AAT[aat_id] ))
        g.add(( AAT[aat_id], CRM.P2_has_type, URIRef(ZAC['aat_object_type']) ))
        object_type_labels = [k for k,v in object_types.items() if v==aat_id]
        if object_type_labels:
          g.add(( AAT[aat_id], RDFS.label, Literal(object_type_labels[0].lower(),lang="it") ))
    for object_type in object_types_value.split(';'):
      object_type_uri_label = create_uri_string(object_type.strip())
      g.add(( URIRef(ZAC[id+'_auction']), CRM.P125_used_objects_of_type, URIRef(ZAC['object_type_'+object_type_uri_label]) ))
      g.add(( URIRef(ZAC['object_type_'+object_type_uri_label]), RDFS.label, Literal(object_type.strip() ) ))
      object_type_aat = [v for k,v in object_types.items() if k==object_type.strip()]
      if object_type_aat:
        g.add(( URIRef(ZAC['object_type_'+object_type_uri_label]), RDFS.seeAlso, AAT[object_type_aat[0]] ))
      g.add(( URIRef(ZAC['object_type_'+object_type_uri_label]), CRM.P2_has_type, URIRef(ZAC['object_type']) ))
  # PERIOD OF OBJECTS
  if pd.notna(object_period):
    # Use the pre-computed timespan_objects (cronologia_to_aat_id_mapping)
    mapped_aat_centuries = timespan_objects.get(object_period, [])
    # Create a URI for the object period based on the object_period_value (the key from the dataframe)
    object_period_uri_label = create_uri_string(object_period)
    g.add(( URIRef(ZAC[id+'_auction']), CRM.P125_used_objects_of_type, URIRef(ZAC['object_period_'+object_period_uri_label]) ))
    g.add(( URIRef(ZAC['object_period_'+object_period_uri_label]), RDFS.label, Literal(object_period) ))
    g.add(( URIRef(ZAC['object_period_'+object_period_uri_label]), CRM.P2_has_type, URIRef(ZAC['object_period']) ))
    for century_aat_id in mapped_aat_centuries:
      if century_aat_id:
        g.add(( URIRef(ZAC['object_period_'+object_period_uri_label]), RDFS.seeAlso, AAT[century_aat_id] ))
        aat_labels = [ "Sec. "+k for k,v in aat_values.items() if v==century_aat_id ]
        if aat_labels:
          g.add(( AAT[century_aat_id], RDFS.label, Literal(aat_labels[0]) ))
  # COLLECTIONS
  g.add(( URIRef(ZAC[id+'_auction']), CRM.P16_used_specific_object, URIRef(ZAC[id]) ))
  if collections_list: # Check if the list is not empty
    for collection in collections_list:
      if collection: # Check if the collection string is not empty
        g.add(( URIRef(ZAC[id+'_auction']), CRM.P16_used_specific_object, URIRef(ZAC[create_uri_string(collection)]) ))
        g.add(( URIRef(ZAC[create_uri_string(collection)]), RDFS.label, Literal(collection) ))
        g.add(( URIRef(ZAC[create_uri_string(collection)]), RDF.type, CRM.E78_Curated_Holding ))
        g.add(( URIRef(ZAC[create_uri_string(collection)]), CRM.P46_is_composed_of, URIRef(ZAC[id+'_lots']) ))
  # ORGANISATION
  if pd.notna(organiser):
    organiser_names = organiser.split("||")
    for organiser_name in organiser_names:
      g.add(( URIRef(ZAC[id+'_auction']), CRM.P9_consists_of, URIRef(ZAC[id+'_organisation']) ))
      g.add(( URIRef(ZAC[id+'_organisation']), RDF.type, CRM.E7_Activity ))
      g.add(( URIRef(ZAC[id+'_organisation']), CRM.P2_has_type, URIRef(ZAC['auction_organisation']) ))
      g.add(( URIRef(ZAC[id+'_organisation']), CRM.P14_carried_out_by, URIRef(ZAC[create_uri_string(organiser_name)]) ))
      g.add(( URIRef(ZAC[create_uri_string(organiser_name)]),RDFS.label, Literal(organiser_name) ))
      g.add(( URIRef(ZAC[create_uri_string(organiser_name)]), RDF.type, CRM.E74_Group ))
      role_assignment(ZAC[id+'_organisation'], organiser_name, ZAC["main_organiser"])
  if pd.notna(secondary_organiser):
    sec_organiser_name = secondary_organiser
    g.add(( URIRef(ZAC[id+'_auction']), CRM.P9_consists_of, URIRef(ZAC[id+'_organisation']) )) # Add an 'organisation' activity for secondary organiser as well
    g.add(( URIRef(ZAC[id+'_organisation']), CRM.P14_carried_out_by, URIRef(ZAC[create_uri_string(sec_organiser_name)]) ))
    g.add(( URIRef(ZAC[create_uri_string(sec_organiser_name)]),RDFS.label, Literal(sec_organiser_name) ))
    g.add(( URIRef(ZAC[create_uri_string(sec_organiser_name)]), RDF.type, CRM.E74_Group ))
    role_assignment(ZAC[id+'_organisation'], sec_organiser_name, ZAC["secondary_organiser"])
  # BATTITORE
  if battitori: # Check if the list is not empty
      for battitore_individual in battitori:
        if battitore_individual: # Check if the individual battitore string is not empty
          battitore_name, role = extract_name_and_place(battitore_individual)
          g.add(( URIRef(ZAC[id+'_auction']), CRM.P9_consists_of, URIRef(ZAC[id+'_auctioneer']) ))
          g.add(( URIRef(ZAC[id+'_auctioneer']), RDF.type, CRM.E7_Activity ))
          g.add(( URIRef(ZAC[id+'_auctioneer']), CRM.P2_has_type, URIRef(ZAC['auctioneering']) ))
          g.add(( URIRef(ZAC[id+'_auctioneer']), CRM.P14_carried_out_by, URIRef(ZAC[create_uri_string(battitore_name)]) ))
          g.add(( URIRef(ZAC[create_uri_string(battitore_name)]), RDFS.label, Literal(battitore_name) ))
          g.add(( URIRef(ZAC[create_uri_string(battitore_name)]), RDF.type, CRM.E21_Person ))
          aat_role = '300025208' # AAT ID for 'auctioneers'
          role_assignment(ZAC[id+'_auctioneer'], battitore_name, AAT[aat_role])
  # OTHER PEOPLE
  if people_involved_list: # Check if the list is not empty
      for person in people_involved_list:
        if person: # Check if the individual string is not empty
          g.add(( URIRef(ZAC[id+'_auction']), CRM.P11_had_participant, URIRef(ZAC[create_uri_string(person)]) ))
          g.add(( URIRef(ZAC[create_uri_string(person)]), RDFS.label, Literal(person) ))
          g.add(( URIRef(ZAC[create_uri_string(person)]), RDF.type, CRM.E21_Person ))
          #aat_role = '300025208' # AAT ID for 'auctioneers'
          role_assignment(ZAC[id+'_auctioneer'], person, ZAC['person_involved'])

  # CODICE ASTA
  if pd.notna(codice_asta):
    g.add((URIRef(ZAC[id+'_auction']), CRM.P48_has_preferred_identifier, URIRef(ZAC[id+'_id']) ))
    g.add((URIRef(ZAC[id+'_id']), RDF.type, CRM.E42_Identifier ))
    g.add((URIRef(ZAC[id+'_id']), RDFS.label, Literal(codice_asta) ))
    g.add((URIRef(ZAC[id+'_id']), CRM.P2_has_type, URIRef(ZAC['codice_asta']) ))
g.serialize('zac_catalogues.ttl')

formatted_label 1908
formatted_label 6 1928/03
formatted_label 1937
formatted_label 1937
formatted_label 1913
formatted_label else 1910


<Graph identifier=N91b39022135442d2baab54435889f8ba (<class 'rdflib.graph.Graph'>)>